# PDF Summarizer Lab Notebook
**Topic:** Transformers and BERT

---

## Learning objectives
Build a PDF summarization pipeline that:
1. Extracts text from a PDF
2. Handles long documents with chunking
3. Summarizes with a local model (Track A) or API model (Track B)
4. Evaluates quality with ROUGE


Runtime tip: set runtime to GPU before starting.

---
# Block 1: Setup and Installation

Install required libraries and verify the runtime environment.

In [ ]:
# Install all required libraries
!pip install -q pymupdf transformers sentencepiece rouge-score tiktoken accelerate

In [ ]:
import fitz
import textwrap
import tiktoken
import torch
from rouge_score import rouge_scorer
from google.colab import files

print("Libraries imported successfully")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

Libraries imported successfully
GPU available: True
Device: Tesla T4


In [ ]:
USE_SAMPLE = True   # Set to False to upload your own PDF

if USE_SAMPLE:
    !wget -q -O sample.pdf https://arxiv.org/pdf/1706.03762  # Attention Is All You Need
    PDF_PATH = "sample.pdf"
    print("Downloaded sample PDF: Attention Is All You Need")
else:
    uploaded = files.upload()
    PDF_PATH = list(uploaded.keys())[0]
    print(f"Uploaded: {PDF_PATH}")

Downloaded sample PDF: Attention Is All You Need


---
# Block 2: PDF Text Extraction
Use PyMuPDF (`fitz`) to extract page-level text, then clean and normalize it.

Discussion: PDFs store content by rendering order, not always reading order. Extraction strategy matters.

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> dict:
    """
    Extract text from each page of a PDF.
    Returns a dict with metadata and a list of page texts.
    """
    doc = fitz.open(pdf_path)

    result = {
        "num_pages": len(doc),
        "pages": []
    }

    for page_num, page in enumerate(doc):
        text = page.get_text("text")  # plain text mode
        result["pages"].append({
            "page": page_num + 1,
            "text": text
        })

    doc.close()
    return result


pdf_data = extract_text_from_pdf(PDF_PATH)
full_text_raw = "\n".join(p['text'] for p in pdf_data['pages'])

print(f"Pages : {pdf_data['num_pages']}")
print("\n--- First 500 chars of page 1 (raw) ---")
print(pdf_data['pages'][0]['text'][:500])
print(f"\nTotal characters: {len(full_text_raw):,}")
print(f"Total words: {len(full_text_raw.split()):,}")

Pages : 15

--- First 500 chars of page 1 (raw) ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K

Total characters: 39,512
Total words: 6,095


In [ ]:
import re

def clean_text(text: str) -> str:
    """
    Basic cleaning:
    - Collapse multiple newlines and spaces
    - Remove page numbers and headers (simple heuristic)
    - Strip leading and trailing whitespace
    """
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


full_text_raw = "\n".join(p['text'] for p in pdf_data['pages'])
full_text = clean_text(full_text_raw)

print("--- Cleaned text (first 500 chars) ---")
print(full_text[:500])
print(f"\nTotal characters: {len(full_text):,}")
print(f"Total words: {len(full_text.split()):,}")

--- Cleaned text (first 500 chars) ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K

Total characters: 39,345
Total words: 6,042


In [ ]:
# Count tokens before selecting a summarization strategy
enc = tiktoken.get_encoding("cl100k_base")  # #https://tiktokenizer.vercel.app/
total_tokens = len(enc.encode(full_text))

print(f"Total tokens: {total_tokens:,}")
print()
print("Model context limits:")
print(f"  BERT          : 512 tokens  -> fits {512/total_tokens*100:.1f}% of the document")
print(f"  BART-large    : 1024 tokens -> fits {1024/total_tokens*100:.1f}% of the document")
print(f"  GPT-3.5       : 4096 tokens -> fits {4096/total_tokens*100:.1f}% of the document")
print(f"  GPT-4 / Claude: 128K tokens -> fits {min(128000/total_tokens*100, 100):.1f}% of the document")
print()
print("Chunking is required for BART on long documents.")

Total tokens: 10,077

Model context limits:
  BERT          : 512 tokens  -> fits 5.1% of the document
  BART-large    : 1024 tokens -> fits 10.2% of the document
  GPT-3.5       : 4096 tokens -> fits 40.6% of the document
  GPT-4 / Claude: 128K tokens -> fits 100.0% of the document

Chunking is required for BART on long documents.


---
# Block 3: Chunking and Summarization

BART have token limits.
For long documents, split text into chunks, summarize each chunk, and combine results.

This follows a map-reduce summarization pattern:

```
Document
  -> Chunk 1 -> Summary 1
  -> Chunk 2 -> Summary 2 -> Final summary
  -> Chunk N -> Summary N
```

In [ ]:
def chunk_text(text: str, max_tokens: int = 800, overlap_tokens: int = 50) -> list[str]:
    """
    Split text into token-aware chunks with overlap.

    overlap_tokens: tokens repeated between adjacent chunks
    to preserve context across boundaries.
    """
    enc = tiktoken.get_encoding("cl100k_base")  #https://tiktokenizer.vercel.app/
    tokens = enc.encode(text)

    chunks = []
    start = 0

    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_body = enc.decode(chunk_tokens)
        chunks.append(chunk_body)
        start += max_tokens - overlap_tokens

    return chunks


chunks = chunk_text(full_text, max_tokens=800, overlap_tokens=50)

print(f"Total chunks: {len(chunks)}")
print("Target chunk size: 800 tokens")
print("Overlap: 50 tokens")
print()
for i, chunk in enumerate(chunks[:3]):
    tok_count = len(enc.encode(chunk))
    print(f"Chunk {i+1}: {tok_count} tokens | preview: {chunk[:80].strip()!r}")

Total chunks: 14
Target chunk size: 800 tokens
Overlap: 50 tokens

Chunk 1: 800 tokens | preview: 'Provided proper attribution is provided, Google hereby grants permission to\nrepr'
Chunk 2: 800 tokens | preview: '24, 15].\nRecurrent models typically factor computation along the symbol position'
Chunk 3: 800 tokens | preview: '.\nFigure 1: The Transformer - model architecture.\nThe Transformer follows this o'


## Choose a track

| Track | Model | Best for |
|---|---|---|
| A | `facebook/bart-large-cnn` | Local/offline workflow |
| B | OpenAI or Anthropic API | Production-style API workflow |

Set `TRACK = "A"` or `TRACK = "B"` below.

In [ ]:
TRACK = "A"  # Change to "B" to use an API-based track

In [ ]:
# Track A implementation: explicit BART model loading
from transformers import BartForConditionalGeneration, BartTokenizer
import torch

model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

def summarize_bart(text: str, max_length: int = 120, min_length: int = 20) -> str:
    inputs = tokenizer(    # BERT specific token
        text,
        return_tensors="pt", # Returns PyTorch tensors instead of Python lists
        max_length=1024, # Limits input size to 1024 tokens
        truncation=True # If input is too long cut it off
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"], # These are the token IDs from tokenizer: "I love AI" → [0, 314, 232, 2]
        max_length=max_length, # Maximum tokens in summary
        min_length=min_length, # Forces summary to be at least this long
        num_beams=4, # This enables beam search, Model tries 4 possible sequences at once and Picks the best one
        early_stopping=True # Stops generation when: all beams produce <eos> (end token)
    )
    return tokenizer.decode(   # Converts tokens to human-readable text
        summary_ids[0], # Output is batch, take first result
        skip_special_tokens=True # Removes tokens like: <s>, </s>, <pad>
        )

print("BART model loaded")
print(summarize_bart(chunks[0]))

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

BART model loaded
The dominant sequence transduction models are based on complex recurrent or Convolutional neural networks that include an encoder and a decoder. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable.


In [ ]:
# # # ============================================================
# # # TRACK B - OpenAI API (swap for Anthropic/Gemini as needed)
# # # ============================================================
# if TRACK == "B":
#     !pip install -q openai
#     from openai import OpenAI
#     from google.colab import userdata

#     # Secret OPENAI_API_KEY
#     client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

#     def api_summarize(text: str, max_words: int = 100) -> str:
#         response = client.chat.completions.create(
#             model="gpt-3.5-turbo",
#             messages=[
#                 {"role": "system",
#                  "content": "You are an expert academic summarizer. Be concise and precise."},
#                 {"role": "user",
#                  "content": f"Summarize the following text in under {max_words} words:\n\n{text}"}
#             ],
#             max_tokens=200
#         )
#         return response.choices[0].message.content.strip()

#     print("OpenAI client ready")
#     print("Chunk 1 summary (sanity check):")
#     print(api_summarize(chunks[0]))

In [ ]:
# ============================================================
# Summarize every chunk
# ============================================================

MAX_CHUNKS_TO_PROCESS = 14

chunk_summaries = []
chunks_to_use = chunks[:MAX_CHUNKS_TO_PROCESS]

print(f"Summarizing {len(chunks_to_use)} chunks...\n")

for i, chunk in enumerate(chunks_to_use):
    print(f"Processing chunk {i+1}/{len(chunks_to_use)} ...", end=" ")

    if TRACK == "A":
        summary = summarize_bart(chunk, max_length=120, min_length=20)
    else:
        pass
        # summary = api_summarize(chunk, max_words=80)

    chunk_summaries.append(summary)
    print("done")

print(f"\nPrepared {len(chunk_summaries)} chunk summaries")

Summarizing 14 chunks...

Processing chunk 1/14 ... done
Processing chunk 2/14 ... done
Processing chunk 3/14 ... done
Processing chunk 4/14 ... done
Processing chunk 5/14 ... done
Processing chunk 6/14 ... done
Processing chunk 7/14 ... done
Processing chunk 8/14 ... done
Processing chunk 9/14 ... done
Processing chunk 10/14 ... done
Processing chunk 11/14 ... done
Processing chunk 12/14 ... done
Processing chunk 13/14 ... done
Processing chunk 14/14 ... done

Prepared 14 chunk summaries


In [ ]:
# ============================================================
# Combine chunk summaries into a final summary
# ============================================================
print("chunk summary 0:",chunk_summaries[0])
print("chunk summary 1:",chunk_summaries[1])
print("chunk summary 13:",chunk_summaries[13])
combined_intermediate = " ".join(chunk_summaries)

print("combined:", combined_intermediate)
print(f"Intermediate combined length: {len(enc.encode(combined_intermediate))} tokens")
print("Running final reduction pass...\n")

if TRACK == "A":
    combined_truncated = enc.decode(enc.encode(combined_intermediate)[:900])
    final_summary = summarize_bart(combined_truncated, max_length=200, min_length=60)
else:
  pass
    # final_summary = api_summarize(combined_intermediate, max_words=200)

print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(textwrap.fill(final_summary, width=80))

chunk summary 0: The dominant sequence transduction models are based on complex recurrent or Convolutional neural networks that include an encoder and a decoder. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable.
chunk summary 1: The Transformer is the first transduction model relying solely on self-attention to compute representations of its input and output. It can reach a new state of the art intranslation quality after being trained for as little as twelve hours on eight P100 GPUs. In the following sections, we will describe the Transformer, motivate viewpoints and discuss its advantages over other models.
chunk summary 13: 5 of 6, apparently involved in anaphora resolution. Note that the attentions are very sharp for this word. Many of the attention heads exhibit behaviour that seems related to the structure of 

---
# Block 4: Evaluation and Discussion

Evaluate summary quality using ROUGE.

| Metric | What it measures |
|---|---|
| ROUGE-1 | Unigram overlap |
| ROUGE-2 | Bigram overlap |
| ROUGE-L | Longest common subsequence |

In [ ]:
# Write a reference summary (1-3 sentences)
REFERENCE_SUMMARY = """The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The Transformer model architecture
dispenses with recurrence and convolutions entirely, relying solely on attention mechanisms.
The model achieves superior translation quality while being more parallelizable and requiring
significantly less time to train."""

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)  # stemming: It reduces words to their root form: running-> run, loved -> love
                                                                                     # Without stemming: "run" ≠ "running" but With stemming: "run" ≈ "running"
# rouge1: Compares single words; Reference: "I love AI"; Generated: "I love ML"; Common words => I, love
# rouge2: Compares pairs of consecutive words
# rougeL (Longest Common Subsequence): Looks at longest matching sequence of words in order

scores = scorer.score(REFERENCE_SUMMARY, final_summary)

print("ROUGE Evaluation Results")
print("-" * 40)
for metric, score in scores.items():
    print(f"  {metric.upper():<10} Precision: {score.precision:.3f}  "
          f"Recall: {score.recall:.3f}  F1: {score.fmeasure:.3f}")

print()
print("Interpretation:")
print("  F1 > 0.4: Good overlap with reference")
print("  F1 > 0.2: Partial overlap ")
print("  F1 < 0.2: Low overlap")

ROUGE Evaluation Results
----------------------------------------
  ROUGE1     Precision: 0.481  Recall: 0.472  F1: 0.476
  ROUGE2     Precision: 0.137  Recall: 0.135  F1: 0.136
  ROUGEL     Precision: 0.288  Recall: 0.283  F1: 0.286

Interpretation:
  F1 > 0.4: Good overlap with reference
  F1 > 0.2: Partial overlap 
  F1 < 0.2: Low overlap
